# ASD Analysis Pipeline

This tutorial demonstrates how to use `SegmentTable` for a batch ASD analysis pipeline. We will crop data for each segment, compute ASDs, and visualize the variation.

In [1]:
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable
from gwpy.timeseries import TimeSeries

def get_synthetic_data():
    return TimeSeries(np.random.randn(1024), sample_rate=64)

segs = [Segment(i*16, i*16+16) for i in range(4)]
st = SegmentTable.from_segments(segs)
st.add_series_column("raw", data=[get_synthetic_data()]*len(st), kind="timeseries")
st

/home/washimi/miniforge3/envs/ws-base/lib/python3.12/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


,span,raw
0,"(0, 16)",<timeseries: 1024 samples>
1,"(16, 32)",<timeseries: 1024 samples>
2,"(32, 48)",<timeseries: 1024 samples>
3,"(48, 64)",<timeseries: 1024 samples>


## Crop and ASD

We can use sugar APIs like `crop()` and `asd()` to process all segments at once.

In [2]:
# Crop to each segment span
st_cropped = st.crop("raw", out_col="cropped")
# Calculate ASD for each segment
st_asd = st_cropped.asd("cropped", out_col="asd", fftlength=2.0)
st_asd.display()

/home/washimi/miniforge3/envs/ws-base/lib/python3.12/site-packages/gwpy/types/series.py:991: UserWarning: TimeSeries.crop given end larger than current end, crop will begin when the Series actually ends.
  warn(


,span,raw,cropped,asd
0,"(0, 16)",<timeseries: 1024 samples>,<timeseries: 1024 samples>,<frequencyseries: 65 bins>
1,"(16, 32)",<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>
2,"(32, 48)",<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>
3,"(48, 64)",<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>


## Multi-channel Summary

You can map custom functions (like calculating band RMS) using `map()`.

In [3]:
def calc_rms(fs):
    return np.sqrt(np.sum(fs.value**2)) # Simplified

st_asd.map("asd", calc_rms, out_col="band_rms", inplace=True)
st_asd.display()

,span,band_rms,raw,cropped,asd
0,"(0, 16)",1.384781,<timeseries: 1024 samples>,<timeseries: 1024 samples>,<frequencyseries: 65 bins>
1,"(16, 32)",0.000000,<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>
2,"(32, 48)",0.000000,<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>
3,"(48, 64)",0.000000,<timeseries: 1024 samples>,<timeseries: 0 samples>,<frequencyseries: 0 bins>
